### Loading Dataset

In [2]:
from datasets import load_dataset

# For demonstration, let's load a large streaming text dataset: 'tiiuae/falcon-refinedweb'.

# Set streaming=True to load data sequentially without loading it all into memory
dataset = load_dataset("tiiuae/falcon-refinedweb", split="train", streaming=True)

print("Streaming the first 10 documents for testing:")
test_sample_list = []
for i, example in enumerate(dataset):
    if i >= 100:
        break
    test_sample_list.append(example)
    print(f"Document {i+1}: {example['content'][:200]}...") # Print the first 200 chars

# Now test_sample_list contains a small list of documents you can use for testing

Resolving data files:   0%|          | 0/1000 [00:00<?, ?it/s]

Streaming the first 10 documents for testing:
Document 1: these birches can be found in many places in Europe - the photos is from a short trip to Baden-Baden in 2007. the clouds in the background are the messengers of the storm Kyrill. here are some more mo...
Document 2: Watch Survivor Redemption Island Season 22 Episode 11: A Mystery Package Online S22e11 Free Stream Megavideo
Article by StreamThatSeries
Horray!time for another dose of very exciting reality series wi...
Document 3: Pesky?
this was a high school project for a president campaign in our government class, yes thats him, for a school project, you guys are crazy
i know his dad from work. very cool and funny guy!!...
Document 4: metalkingdom.net [ 80′s @ 8 Feature Video – Big City Nights [VIDEO] By Chris Chapman March 13, 2012 It's time to rock out to those talented Germans Rudolf and Klaus from the Scorpions! "Big City Night...
Document 5: Splice Review
Black Ops Escalation Map Pack [VIDEO]
Scream 4 Review-No Spoilers
Bes

In [3]:
test_sample_list[2]

{'content': 'Pesky?\nthis was a high school project for a president campaign in our government class, yes thats him, for a school project, you guys are crazy\ni know his dad from work. very cool and funny guy!!',
 'url': 'http://101squadron.com/blog/2007/05/pesky-peculiarities-of-css.html/comment-page-1',
 'timestamp': datetime.datetime(2013, 5, 18, 10, 21, 35),
 'dump': 'CC-MAIN-2013-20',
 'segment': '1368696382261',
 'image_urls': [['http://101squadron.com/uploaded_images/Conger_89_cal-757679.jpg',
   None],
  ["http://101squadron.com/uploaded_images/Hammerin'-Hank-Conger-795118.jpg",
   None],
  ['http://101squadron.com/uploaded_images/hconger06108180bm-753187.jpg',
   None],
  ['http://101squadron.com/uploaded_images/HHMex8Au-794539.jpg', None],
  ['http://101squadron.com/uploaded_images/Conger-back-712123.jpg', None]]}

### Design Principles (so that methods can scale to trillion token datasets)


Coode should be written to be:

Stateless

Single-pass

Map-friendly

Auditable

Cheap-first, expensive-last

Signals-first, bands-later

### Overall Pipeline - High-level structure

```
raw_text
  ↓
normalize + tokenize (ONCE)
  ↓
cheap structural signals
  ↓
regex modality signals
  ↓
token-level stats
  ↓
(optional) readability
  ↓
signals dict  ← persisted
  ↓
band_assignment(signals, thresholds_version)
  ↓
band + decision metadata
```


### ========== CONFIGURATION ==========

In [22]:
SUPPORTED_DATASETS = {"dolma", "fineweb", "sangraha","falcon-refinedweb"}

# Unicode ranges for Indic scripts
INDIC_SCRIPT_RANGES = [
    (0x0900, 0x097F),  # Devanagari (Hindi, Marathi, Sanskrit)
    (0x0980, 0x09FF),  # Bengali
    (0x0A00, 0x0A7F),  # Gurmukhi (Punjabi)
    (0x0A80, 0x0AFF),  # Gujarati
    (0x0B00, 0x0B7F),  # Oriya
    (0x0B80, 0x0BFF),  # Tamil
    (0x0C00, 0x0C7F),  # Telugu
    (0x0C80, 0x0CFF),  # Kannada
    (0x0D00, 0x0D7F),  # Malayalam
]

# Domain mapping
DOMAIN_MAPPING = {
    "dolma": {
        "cc_en_head": "general_web_clean",
        "cc_en_middle": "general_web_clean",
        "stack": "code_repos",
        "arxiv": "math_science",
        "wiki": "encyclopedic",
        "c4": "general_web_clean",
    },
    "fineweb": "general_web_clean",
    "sangraha": "general_web_clean",
    "falcon-refinedweb": "general_web_clean"
}

# Band difficulty centroids
BAND_DIFFICULTY = {
    "B0": 0.10,
    "B1": 0.22,
    "B2": 0.40,
    "B3": 0.60,
    "B4": 0.78,
    "B5": 0.92,
}

# Floors and caps
BAND_FLOORS = {
    "B0": 0.10,
    "B1": 0.14,
    "B2": 0.18,
    "B3": 0.14,
    "B4": 0.06,
    "B5": 0.02,
}

BAND_CAPS = {}

# Stage-specific constraints
STAGE_CONSTRAINTS = {
    "1B": {
        "indic_allowed": False,
        "agentic_allowed_bands": [],
        "cot_caps_per_band": {"B3": 0.03, "B4": 0.06, "B5": 0.08},
    },
    "3B": {
        "indic_allowed": True,
        "agentic_allowed_bands": [],
        "cot_caps_per_band": {"B3": 0.04, "B4": 0.07, "B5": 0.09},
    },
    "8B": {
        "indic_allowed": True,
        "agentic_allowed_bands": ["B4", "B5"],
        "cot_caps_per_band": {"B3": 0.05, "B4": 0.08, "B5": 0.10},
    },
    "70B": {
        "indic_allowed": True,
        "agentic_allowed_bands": ["B4", "B5"],
        "cot_caps_per_band": {"B3": 0.05, "B4": 0.08, "B5": 0.10},
    },
}

# Thresholds
THRESHOLDS_V1 = {
    "B0": {"max_tokens": 80},
    "B1": {"readability": 6},
    "B2": {"readability": 10},
    "B3": {"readability": 14},
    "B4": {"has_math": True},
    "B5": {"has_agentic": True}
}

### Precompiled, reusable components

In [5]:
import re
from functools import lru_cache
from collections import Counter

# ---------- Precompiled regex ----------
RE_CODE = re.compile(r"```|def\s+\w+\(|class\s+\w+|#include\s+<|function\s+\w+\(")
RE_COT = re.compile(
    r"let'?s think step by step|step by step|first[, ]|second[, ]|therefore|reasoning:",
    re.IGNORECASE
)
RE_AGENTIC = re.compile(
    r"Thought:|Action:|Observation:|\"tool\"\s*:|\"arguments\"\s*:",
    re.IGNORECASE
)
RE_MATH = re.compile(r"[∑∫√≈≠≤≥→∞]")
RE_RESEARCH_PAPER = re.compile(
    r"\bAbstract[:\s]|"
    r"\bReferences[:\s]|"
    r"\b(?:arXiv|doi):\s*\d|"
    r"\bet al\.|"
    r"\[[\d,\s]+\].*\[[\d,\s]+\]",
    re.IGNORECASE
)

### Language Detection

In [6]:
def detect_language_fast(text: str, metadata_lang: str = None) -> str:
    """
    Fast language detection: metadata first, then heuristic.

    Returns: 'en', 'indic', or 'other'
    """
    if metadata_lang:
        if metadata_lang.startswith('en'):
            return 'en'
        if metadata_lang in ['hi', 'bn', 'ta', 'te', 'mr', 'gu', 'kn', 'ml', 'pa', 'or']:
            return 'indic'

    for char in text[:500]:
        code = ord(char)
        for start, end in INDIC_SCRIPT_RANGES:
            if start <= code <= end:
                return 'indic'

    return 'en'

### METADATA EXTRACTION

In [23]:
def extract_metadata(sample: dict, dataset_name: str) -> dict:
    """
    Extract standardized metadata from different dataset formats.
    """
    if dataset_name not in SUPPORTED_DATASETS:
        raise NotImplementedError(
            f"Dataset '{dataset_name}' is not supported. "
            f"Please implement metadata extraction for this dataset. "
            f"Supported datasets: {SUPPORTED_DATASETS}"
        )

    metadata = {
        "source_dataset": dataset_name,
        "doc_id": None,
        "url": None,
        "language": None,
        "text": None,
        "source_subsection": None
    }

    if dataset_name == "dolma":
        metadata["doc_id"] = sample.get("id")
        metadata["text"] = sample.get("text")
        metadata["source_subsection"] = sample.get("source")

    elif dataset_name == "fineweb":
        metadata["doc_id"] = sample.get("id")
        metadata["text"] = sample.get("text")
        metadata["url"] = sample.get("url")
        metadata["language"] = sample.get("language")

    elif dataset_name == "falcon-refinedweb":
        metadata["doc_id"] = sample.get("segment")
        metadata["text"] = sample.get("content")
        metadata["url"] = sample.get("url")
        metadata["language"] = sample.get("language")

    elif dataset_name == "sangraha":
        metadata["text"] = sample.get("text") or sample.get("content")
        metadata["doc_id"] = sample.get("id")

    # Detect language
    metadata["language"] = detect_language_fast(
        metadata["text"],
        metadata.get("language")
    )

    return metadata

### Single-pass normalization & tokenization

In [8]:
def normalize_and_tokenize(text: str):
    """Pure, stateless, single-pass normalization."""
    text = text.strip()
    sentences = re.split(r"[.!?]+", text)
    tokens = re.findall(r"\b\w+\b", text.lower())
    return {
        "text": text,
        "sentences": sentences,
        "tokens": tokens,
        "num_chars": len(text),
        "num_tokens": len(tokens),
        "num_sentences": max(1, len(sentences))
    }

### Cheap structural signals

In [9]:
def extract_structural_signals(doc):
    """Very cheap signals. Enables early exits."""
    return {
        "num_tokens": doc["num_tokens"],
        "avg_sentence_length": doc["num_tokens"] / doc["num_sentences"]
    }

### Regex-based modality detection

In [10]:
def extract_modality_signals(text: str):
    """Regex-only, cheap, high-precision modality detection."""
    return {
        "has_code": bool(RE_CODE.search(text)),
        "has_cot": bool(RE_COT.search(text)),
        "has_agentic": bool(RE_AGENTIC.search(text)),
        "has_math": bool(RE_MATH.search(text)),
        "has_research_paper": bool(RE_RESEARCH_PAPER.search(text))
    }

### Approximate token statistics

In [11]:
def extract_token_stats(tokens):
    """Approximate lexical difficulty metrics."""
    if not tokens:
        return {"rare_ratio": 0.0}

    freq = Counter(tokens)
    rare = sum(1 for t in tokens if freq[t] == 1)

    return {
        "rare_ratio": rare / len(tokens)
    }

### Optional readability (expensive, last)

In [12]:
@lru_cache(maxsize=100_000)
def _syllables(word: str):
    return max(1, len(re.findall(r"[aeiouy]+", word)))

def flesch_kincaid_grade(tokens, sentences):
    if len(tokens) < 10:
        return 0.0

    syllables = sum(_syllables(t) for t in tokens)
    return (
        0.39 * (len(tokens) / max(1, len(sentences))) +
        11.8 * (syllables / len(tokens)) -
        15.59
    )

### Signal extraction orchestrator

In [13]:
def extract_signals(text: str):
    """Single entry point. Signals-first by design."""
    doc = normalize_and_tokenize(text)

    if doc["num_tokens"] < 40:
        return {
            "signals": {"num_tokens": doc["num_tokens"]},
            "early_band": "B0"
        }

    signals = {}
    signals.update(extract_structural_signals(doc))
    signals.update(extract_modality_signals(doc["text"]))
    signals.update(extract_token_stats(doc["tokens"]))

    if (
        signals["has_code"]
        or signals["has_math"]
        or signals["avg_sentence_length"] > 18
    ):
        signals["readability"] = flesch_kincaid_grade(
            doc["tokens"], doc["sentences"]
        )
    else:
        signals["readability"] = 0.0

    return {
        "signals": signals,
        "early_band": None
    }

### Band assignment logic (separate, versioned)

In [14]:
def assign_band(signals, thresholds=THRESHOLDS_V1):
    """Pure decision logic. No text access."""
    if signals.get("has_agentic"):
        return "B5"

    if signals.get("has_math") and signals.get("readability", 0) > 12:
        return "B4"

    if signals.get("has_code"):
        return "B3"

    r = signals.get("readability", 0)
    if r > thresholds["B3"]["readability"]:
        return "B3"
    if r > thresholds["B2"]["readability"]:
        return "B2"
    if r > thresholds["B1"]["readability"]:
        return "B1"

    return "B0"

### Modality Assignment

In [15]:
def assign_modality(signals: dict) -> str:
    """Assign primary modality based on signal hierarchy."""
    if signals.get("has_agentic"):
        return "agentic_traces"

    if signals.get("has_cot"):
        return "cot_reasoning"

    if signals.get("has_research_paper"):
        return "research_papers"

    if signals.get("has_code"):
        return "code"

    if signals.get("has_math"):
        return "math"

    return "general_text"

### Domain Assignment

In [16]:
def assign_domain(metadata: dict, signals: dict) -> str:
    """Assign domain group based on source metadata and content signals."""
    dataset = metadata["source_dataset"]

    if dataset == "dolma":
        subsource = metadata.get("source_subsection")
        return DOMAIN_MAPPING["dolma"].get(subsource, "general_web_clean")

    return DOMAIN_MAPPING.get(dataset, "general_web_clean")

### Stage Constraints

In [17]:
def check_stage_constraints(classification: dict, stage: str) -> dict:
    """Check if document violates stage-specific constraints."""
    if classification.get("rejected"):
        return classification

    constraints = STAGE_CONSTRAINTS[stage]
    band = classification["band"]
    modality = classification["modality"]
    language = classification["metadata"]["language"]

    # Check Indic language constraint
    if language == "indic" and not constraints["indic_allowed"]:
        classification["rejected"] = True
        classification["rejection_reason"] = f"indic_not_allowed_at_{stage}"
        return classification

    # Check agentic constraint
    if modality == "agentic_traces" and band not in constraints["agentic_allowed_bands"]:
        classification["rejected"] = True
        classification["rejection_reason"] = f"agentic_not_allowed_in_{band}_at_{stage}"
        return classification

    return classification

### Final map-friendly band classification API

In [18]:
def classify_document(sample: dict, dataset_name: str, stage: str = None, threshold_version="v1"):
    """
    Map-friendly, stateless classification with full metadata, modality, domain, and constraint checking.

    Args:
        sample: Raw sample dict from dataset
        dataset_name: Source dataset identifier
        stage: Training stage ("1B", "3B", "8B", "70B") - optional
        threshold_version: Version of thresholds to use

    Returns:
        Dict with band, modality, domain, signals, metadata, and rejection info
    """
    metadata = extract_metadata(sample, dataset_name)

    # Filter out non-English/non-Indic languages
    if metadata["language"] == "other":
        return {
            "band": None,
            "modality": None,
            "domain": None,
            "rejected": True,
            "rejection_reason": "language_not_en_or_indic",
            "metadata": metadata,
            "signals": None,
            "threshold_version": threshold_version
        }

    text = metadata["text"]
    result = extract_signals(text)

    # Determine band
    if result["early_band"] is not None:
        band = result["early_band"]
    else:
        band = assign_band(result["signals"])

    # Assign modality and domain
    modality = assign_modality(result["signals"])
    domain = assign_domain(metadata, result["signals"])

    classification = {
        "band": band,
        "modality": modality,
        "domain": domain,
        "rejected": False,
        "signals": result["signals"],
        "metadata": metadata,
        "threshold_version": threshold_version
    }

    # Apply stage-specific constraints if stage is provided
    if stage:
        classification = check_stage_constraints(classification, stage)

    return classification

In [24]:
bands = [classify_document(doc,"falcon-refinedweb") for doc in test_sample_list]

In [ ]:
bands[:10]

[{'band': 'B0',
  'signals': {'num_tokens': 96,
   'avg_sentence_length': 12.0,
   'has_code': False,
   'has_cot': False,
   'has_agentic': False,
   'has_math': False,
   'rare_ratio': 0.5520833333333334,
   'readability': 0.0},
  'threshold_version': 'v1'},
 {'band': 'B2',
  'signals': {'num_tokens': 1918,
   'avg_sentence_length': 23.10843373493976,
   'has_code': False,
   'has_cot': True,
   'has_agentic': False,
   'has_math': False,
   'rare_ratio': 0.2601668404588113,
   'readability': 12.352737540359566},
  'threshold_version': 'v1'},
 {'band': 'B0', 'signals': {'num_tokens': 37}, 'threshold_version': 'v1'},
 {'band': 'B0',
  'signals': {'num_tokens': 94,
   'avg_sentence_length': 13.428571428571429,
   'has_code': False,
   'has_cot': True,
   'has_agentic': False,
   'has_math': False,
   'rare_ratio': 0.48936170212765956,
   'readability': 0.0},
  'threshold_version': 'v1'},
 {'band': 'B0', 'signals': {'num_tokens': 35}, 'threshold_version': 'v1'},
 {'band': 'B0',
  'signa

### calculating band distribution

For each model size (1B → 70B), this code decides what fraction of training data should come from each difficulty band, based on:

**how hard that band is**,

**how capable the model is**,

**how much of that band naturally exists in the data**,

and **some safety rules to prevent bad behavior**.

In [25]:
def map_band_tokens(sample: dict, dataset_name: str, stage: str = None) -> dict:
    """MAP STEP: Classifies band and extracts token count."""
    result = classify_document(sample, dataset_name, stage)

    band = result["band"]
    num_tokens = result["signals"].get("num_tokens", 0) if result["signals"] else 0

    return {
        "band": band,
        "modality": result["modality"],
        "domain": result["domain"],
        "tokens": num_tokens,
        "rejected": result["rejected"],
        "metadata": result["metadata"]
    }

In [26]:
from collections import defaultdict
from typing import Iterable, Dict

def reduce_base_distribution(mapped_rows: Iterable[dict]) -> dict:
    """REDUCE STEP: Aggregates token counts per band."""
    band_token_totals = defaultdict(int)
    total_tokens = 0
    rejected_count = 0

    for row in mapped_rows:
        if row["rejected"]:
            rejected_count += 1
            continue

        band = row["band"]
        tokens = row["tokens"]

        band_token_totals[band] += tokens
        total_tokens += tokens

    if total_tokens == 0:
        raise ValueError("Total token count is zero; invalid dataset.")

    base_distribution = {
        band: band_token_totals[band] / total_tokens
        for band in band_token_totals
    }

    return {
        'band_dist': base_distribution,
        'total_tokens': total_tokens,
        'rejected_count': rejected_count
    }


In [27]:
def compute_base_distribution(samples: Iterable[dict], dataset_name: str, stage: str = None) -> dict:
    """Convenience wrapper: MAP + REDUCE"""
    mapped = (map_band_tokens(s, dataset_name, stage) for s in samples)
    return reduce_base_distribution(mapped)

In [28]:
base_distribution = compute_base_distribution(test_sample_list,"falcon-refinedweb")
base_distribution

{'band_dist': {'B0': 0.5732323232323232,
  'B2': 0.13295708765507422,
  'B3': 0.23922106975798252,
  'B1': 0.030167446274828823,
  'B5': 0.0244220730797912},
 'total_tokens': 59004,
 'rejected_count': 0}

### Combining base distributions from multiple datasets

In [29]:
def combine_base_distributions(distributions: list[dict]) -> dict:
    """Combine multiple base distributions weighted by their token counts."""
    band_token_totals = defaultdict(int)
    grand_total_tokens = 0

    for dist_info in distributions:
        band_dist = dist_info['band_dist']
        total_tokens = dist_info['total_tokens']

        for band, proportion in band_dist.items():
            band_token_totals[band] += proportion * total_tokens

        grand_total_tokens += total_tokens

    if grand_total_tokens == 0:
        raise ValueError("Total tokens across all distributions is zero")

    combined = {
        band: tokens / grand_total_tokens
        for band, tokens in band_token_totals.items()
    }

    return combined

### Band proportion calculations

### Band difficulty centroids (quantile based)

### Capacity function (scaling theory)

In [30]:
def model_capacity(params: float, min_params: float = 1e9, max_params: float = 70e9) -> float:
    """Normalized model capacity based on log(parameter count)."""
    return (
        math.log(params) - math.log(min_params)
    ) / (
        math.log(max_params) - math.log(min_params)
    )

### Capacity–difficulty alignment function

In [31]:
def alignment_weight(difficulty: float, capacity: float, lambda_align: float = 3.0) -> float:
    """Smooth alignment between band difficulty and model capacity."""
    return math.exp(-lambda_align * abs(difficulty - capacity))


### Raw weight computation

In [32]:
def raw_band_weights(base_distribution: Dict[str, float], params: float, lambda_align: float = 3.0) -> Dict[str, float]:
    """Compute raw (unnormalized) band weights."""
    capacity = model_capacity(params)

    weights = {}
    for band, base_w in base_distribution.items():
        d_b = BAND_DIFFICULTY[band]
        align = alignment_weight(d_b, capacity, lambda_align)
        weights[band] = base_w * align

    return weights

### Apply floors and caps (policy enforcement)

In [33]:
def apply_floors_and_caps(weights: Dict[str, float], floors: Dict[str, float], caps: Dict[str, float] | None = None) -> Dict[str, float]:
    """Enforce curriculum floors and caps."""
    caps = caps or {}
    constrained = {}

    for band, w in weights.items():
        w_floor = max(w, floors.get(band, 0.0))
        w_cap = min(w_floor, caps.get(band, float("inf")))
        constrained[band] = w_cap

    return constrained

### Renormalization

In [34]:
def renormalize(weights: Dict[str, float]) -> Dict[str, float]:
    """Normalize weights to sum to 1.0."""
    total = sum(weights.values())
    if total == 0:
        raise ValueError("Cannot renormalize: total weight is zero.")
    return {b: w / total for b, w in weights.items()}


### End-to-end API

In [35]:
def compute_band_proportions(base_distribution: Dict[str, float], params: float, lambda_align: float = 3.0) -> Dict[str, float]:
    """Full curriculum band proportion computation for one training stage."""
    # Ensure all bands present
    for band in ["B0", "B1", "B2", "B3", "B4", "B5"]:
        if band not in base_distribution:
            base_distribution[band] = 0.0

    raw = raw_band_weights(
        base_distribution=base_distribution,
        params=params,
        lambda_align=lambda_align
    )

    constrained = apply_floors_and_caps(
        raw,
        floors=BAND_FLOORS,
        caps=BAND_CAPS
    )

    final = renormalize(constrained)
    return final

### Proportions obtained

In [39]:
import math
from typing import Dict

In [36]:
for band in ["B0", "B1", "B2", "B3", "B4", "B5"]:
  if not band in base_distribution:
    base_distribution[band] = 0

In [41]:
STAGES = {
    "1B": 1e9,
    "3B": 3e9,
    "8B": 8e9,
    "70B": 70e9,
}

for stage, params in STAGES.items():
    proportions = compute_band_proportions(
        base_distribution=base_distribution['band_dist'],
        params=params
    )
    print(stage, proportions)

1B {'B0': 0.44021782974755996, 'B2': 0.18659405675081334, 'B3': 0.1451287108061882, 'B1': 0.1451287108061882, 'B5': 0.020732672972312596, 'B4': 0.06219801891693778}
3B {'B0': 0.3974638325695162, 'B2': 0.2008453891434946, 'B3': 0.15621308044494026, 'B1': 0.15621308044494026, 'B5': 0.022316154349277178, 'B4': 0.06694846304783153}
8B {'B0': 0.23763594146980338, 'B2': 0.24003064140024463, 'B3': 0.2289626331963196, 'B1': 0.18669049886685696, 'B5': 0.02667007126669385, 'B4': 0.08001021380008154}
70B {'B0': 0.15625, 'B2': 0.28125, 'B3': 0.21875000000000003, 'B1': 0.21875000000000003, 'B5': 0.03125, 'B4': 0.09375}
